### RMRB Analysis
Tech
- Text extraction with "广告": full-page ad
- Cipher text extraction with encoded string "!!"!": half-page ad with "·广告·" text
- OCR image recognition: half-page ad with no "·广告·" text, but image "广告"

Target
- Find all advertisements in RMRB, including full-page ads and half-page ads
  - Full-page
    - Image format: OCR tech
    - Text format: OCR tech -> Text recognition
      - or Text extraction -> decode(hard)
  - Half-page: image extraction(CMYK and RGB conversion)(hard)
- Categorize all ads by industry

RMRB Analysis v0
- Build name function `PATH_FUN`, for naming conventions and convenience
- Make pdf check function `Check_PDF_Exist`, find the missing PDF(s)
  - `Generate_Dates`: Generate dates of a whole year
- Construct the core function `Genetare_AD_Image`'s basic component
  - `Check_Folder_Exist`: Check existence of folder
  - Test three years' data: 2022, 2021, 2020
  - **Time consumed: More than 2h for a single year**
> Note: the `Genetare_AD_Image` now can only recognize a page contains plaintext "广告", except from ciphertext "广告" and image element "广告"

In [60]:
from PyPDF2 import PdfReader, PdfWriter
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
from datetime import datetime, timedelta
from pathlib import Path
import os

from pdfminer.pdfdocument import PDFDocument
from pdfminer.pdfparser import PDFParser
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.converter import PDFPageAggregator
from pdfminer.layout import LTTextBoxHorizontal, LAParams
from pdfminer.high_level import extract_text
# from pdfminer.pdfinterp import PDFTextExtractionNotAllowed

# Set the path to the Tesseract installation
pytesseract.pytesseract.tesseract_cmd = "D:\\Tesseract-OCR\\tesseract.exe"

In [2]:
Advertisement = "广告"

In [49]:
ROOT_PATH = "D:/AI_data_analysis/RMRB/"
RMRB2024090616 = "D:/AI_data_analysis/RMRB/2024090616.pdf"
def PATH_FUN(YEAR, MONTH, DAY, EDITION, SUFFIX, NAME_bool=False):
    Split_Year = ["2023"] 
    NAME = YEAR + MONTH + DAY
    INFO_PATH = (
        NAME 
        + (
            ("/" + NAME + EDITION) 
            if YEAR in Split_Year else ""
        ))
    SUFFIX = ".pdf"
    pdf_path = ROOT_PATH + YEAR + "/" + INFO_PATH + SUFFIX
    if NAME_bool:
        return pdf_path, NAME
    else: return  pdf_path
PDF_PATH = PATH_FUN("2022", "01", "02", "02", ".pdf")
PDF_PATH

'D:/AI_data_analysis/RMRB/2022/20220102.pdf'

In [ ]:
def Generate_Dates(year):
    start_date = datetime(year, 1, 1)
    end_date = datetime(year + 1, 1, 1)
    current_date = start_date
    
    dates = []
    while current_date < end_date:
        dates.append(current_date.strftime('%Y%m%d'))
        current_date += timedelta(days=1)
    return dates

def Check_PDF_Exist(YEAR):
    MISSING_PDF = []
    start_date = datetime(int(YEAR), 1, 1)
    end_date = datetime(int(YEAR) + 1, 1, 1)
    current_date = start_date
    while current_date < end_date:
        MONTH = str(current_date.month)
        MONTH = ("0" + MONTH) if len(MONTH) == 1 else MONTH
        DAY = str(current_date.day)
        DAY = ("0" + DAY) if len(DAY) == 1 else DAY
        PATH = PATH_FUN(YEAR, MONTH, DAY, "", ".pdf")
        try:
            with open(PATH, "rb") as file:
                PdfReader(file)
        except FileNotFoundError: 
            print(PATH)
            MISSING_PDF.append(PATH)
        current_date += timedelta(days=1)
        return MISSING_PDF
# Check_PDF_Exist("2019")
"""
D:/AI_data_analysis/RMRB/2018/20180216.pdf
D:/AI_data_analysis/RMRB/2018/20180425.pdf
D:/AI_data_analysis/RMRB/2018/20180516.pdf
D:/AI_data_analysis/RMRB/2018/20180726.pdf
D:/AI_data_analysis/RMRB/2018/20180817.pdf
D:/AI_data_analysis/RMRB/2018/20180818.pdf
D:/AI_data_analysis/RMRB/2018/20181012.pdf

D:/AI_data_analysis/RMRB/2019/20190107.pdf
D:/AI_data_analysis/RMRB/2019/20190402.pdf
D:/AI_data_analysis/RMRB/2019/20190410.pdf
D:/AI_data_analysis/RMRB/2019/20190412.pdf
D:/AI_data_analysis/RMRB/2019/20190728.pdf
D:/AI_data_analysis/RMRB/2019/20190802.pdf

"""

In [64]:
def Count_Files(folder_path):
    # Create a Path object for the folder
    folder = Path(folder_path)
    # Count all files in the folder
    file_count = sum(1 for file in folder.iterdir() if file.is_file())
    return file_count
# Count_Files("D:/AI_data_analysis/RMRB")

def Check_Folder_Exist(folder_path):
    # Check if the folder exists
    if not os.path.exists(folder_path):
        # Create the folder if it does not exist
        os.makedirs(folder_path)
        print(f'Folder created: {folder_path}')
    else:
        print(f'Folder already exists: {folder_path}')
# Check_Folder_Exist("D:/AI_data_analysis/RMRB")

In [72]:
# Generate image for all ad pages using OCR
def Genetare_AD_Image(YEAR):
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD"
    Check_Folder_Exist(FOLD_PATH)
    start_date = datetime(int(YEAR), 1, 1)
    end_date = datetime(int(YEAR) + 1, 1, 1)
    current_date = start_date
    while current_date < end_date:
        MONTH = str(current_date.month)
        MONTH = ("0" + MONTH) if len(MONTH) == 1 else MONTH
        DAY = str(current_date.day)
        DAY = ("0" + DAY) if len(DAY) == 1 else DAY
        PDF_PATH, NAME = PATH_FUN(YEAR, MONTH, DAY, "", ".pdf", NAME_bool=True)
        print("PDF PATH:", PDF_PATH)
        with open(PDF_PATH, "rb") as file:
            PDF = PdfReader(file)
            PAGES_NUM = len(PDF.pages)
            IMAGE = convert_from_path(PDF_PATH)
            for page_num in range(PAGES_NUM):
                text = PDF.pages[page_num].extract_text().replace(" ", "")
                if Advertisement in text: # if "广告" appears in the text
                    IMAGE_PATH = FOLD_PATH + "/" + f"{NAME}_{page_num+1}_AD.png"
                    IMAGE[page_num].save(IMAGE_PATH, "PNG")
                    print(IMAGE_PATH)
        current_date += timedelta(days=1)
Genetare_AD_Image("2022")

PDF PATH: D:/AI_data_analysis/RMRB/2022/20220101.pdf
D:/AI_data_analysis/RMRB/2022_AD/20220101_8_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2022/20220102.pdf
D:/AI_data_analysis/RMRB/2022_AD/20220102_7_AD.png
D:/AI_data_analysis/RMRB/2022_AD/20220102_8_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2022/20220103.pdf
D:/AI_data_analysis/RMRB/2022_AD/20220103_2_AD.png
D:/AI_data_analysis/RMRB/2022_AD/20220103_8_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2022/20220104.pdf
D:/AI_data_analysis/RMRB/2022_AD/20220104_8_AD.png
D:/AI_data_analysis/RMRB/2022_AD/20220104_15_AD.png
D:/AI_data_analysis/RMRB/2022_AD/20220104_16_AD.png
D:/AI_data_analysis/RMRB/2022_AD/20220104_19_AD.png
D:/AI_data_analysis/RMRB/2022_AD/20220104_20_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2022/20220105.pdf
D:/AI_data_analysis/RMRB/2022_AD/20220105_8_AD.png
D:/AI_data_analysis/RMRB/2022_AD/20220105_20_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2022/20220106.pdf
D:/AI_data_analysis/RMRB/2022_AD/20220106_5_AD.png
D:/AI_data_ana

In [73]:
Genetare_AD_Image("2021")

Folder created: D:/AI_data_analysis/RMRB/2021_AD
PDF PATH: D:/AI_data_analysis/RMRB/2021/20210101.pdf
D:/AI_data_analysis/RMRB/2021_AD/20210101_7_AD.png
D:/AI_data_analysis/RMRB/2021_AD/20210101_8_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2021/20210102.pdf
D:/AI_data_analysis/RMRB/2021_AD/20210102_8_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2021/20210103.pdf
D:/AI_data_analysis/RMRB/2021_AD/20210103_6_AD.png
D:/AI_data_analysis/RMRB/2021_AD/20210103_8_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2021/20210104.pdf
D:/AI_data_analysis/RMRB/2021_AD/20210104_8_AD.png
D:/AI_data_analysis/RMRB/2021_AD/20210104_20_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2021/20210105.pdf
D:/AI_data_analysis/RMRB/2021_AD/20210105_15_AD.png
D:/AI_data_analysis/RMRB/2021_AD/20210105_16_AD.png
D:/AI_data_analysis/RMRB/2021_AD/20210105_17_AD.png
D:/AI_data_analysis/RMRB/2021_AD/20210105_20_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2021/20210106.pdf
D:/AI_data_analysis/RMRB/2021_AD/20210106_7_AD.png
D:/AI_data_analy

In [74]:
Genetare_AD_Image("2020")

Folder created: D:/AI_data_analysis/RMRB/2020_AD
PDF PATH: D:/AI_data_analysis/RMRB/2020/20200101.pdf
D:/AI_data_analysis/RMRB/2020_AD/20200101_6_AD.png
D:/AI_data_analysis/RMRB/2020_AD/20200101_7_AD.png
D:/AI_data_analysis/RMRB/2020_AD/20200101_8_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2020/20200102.pdf
D:/AI_data_analysis/RMRB/2020_AD/20200102_8_AD.png
D:/AI_data_analysis/RMRB/2020_AD/20200102_20_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2020/20200103.pdf
D:/AI_data_analysis/RMRB/2020_AD/20200103_8_AD.png
D:/AI_data_analysis/RMRB/2020_AD/20200103_20_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2020/20200104.pdf
D:/AI_data_analysis/RMRB/2020_AD/20200104_7_AD.png
D:/AI_data_analysis/RMRB/2020_AD/20200104_8_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2020/20200105.pdf
D:/AI_data_analysis/RMRB/2020_AD/20200105_8_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2020/20200106.pdf
D:/AI_data_analysis/RMRB/2020_AD/20200106_10_AD.png
D:/AI_data_analysis/RMRB/2020_AD/20200106_20_AD.png
PDF PATH: D:/AI_d